# M1/M2 実行報告・参照報告

最終更新: 2026-07-20 UTC

## 結論

| milestone | run ID | 判定 | checkpoint | status |
|---|---|---|---:|---|
| M1 | `M1-20260719T235423Z-a7cacde2ead9` | M2へ進行可 | 14 | `NOT_CERTIFIED` |
| M2 | `M2-20260720T005145Z-dd3e385d0a61` | M3探索実装へ進行可 | 14 | `NOT_CERTIFIED` |

M2 は `j2_max=1` の六脚4D link-starについて、独立に構成した dense Haar projector と armillary projector が全64 sectorで厳密一致することを確認しました。これはM3の探索的GPU実装を開始する根拠になりますが、4D RG、mass gap、熱力学極限、連続極限の証明ではありません。


## M2 実行結果

- acceptance gate: 14/14 PASS
- exact dense generator residual: 64/64 zero
- dense–armillary symbolic matrix equality: 64/64
- gauge-noninvariant odd-half sector: 32/32 zero
- armillary isometry: 64/64 exact
- cubic symmetry: 48 actions、20 canonical sectors
- Wigner cache: exact deterministic regeneration PASS
- tensor checkpoint shards: 64
- CPU tests: 48 passed、3 GPU tests deselected
- fresh-process resume: PASS
- newest checkpoint corruption fallback: PASS
- heuristic result: なし
- checkpoint: 168,147 bytes、save 0.464 s、verify 0.004 s
- peak CPU memory: 約369 MiB
- GPU memory: 0（M2の証明経路はCPU exact arithmetic）

### 重要な制限

float64 checkpoint shard は再開確認用であり、証明上の bound ではありません。M3 の RSVD、特異値、influence proxy も探索値であり、`CERTIFIED` の根拠にはできません。


## 参照先

### 実行 artifact

- M1 report: `/storage/validated_4d_su2_rg/runs/M1-20260719T235423Z-a7cacde2ead9/reports/M1_report.json`
- M2 report: `/storage/validated_4d_su2_rg/runs/M2-20260720T005145Z-dd3e385d0a61/reports/M2_report.json`
- M2 acceptance: `/storage/validated_4d_su2_rg/runs/M2-20260720T005145Z-dd3e385d0a61/reports/M2_acceptance.json`
- M2 latest checkpoint: `/storage/validated_4d_su2_rg/runs/M2-20260720T005145Z-dd3e385d0a61/checkpoints/ckpt_000014`

### 判定と仕様

- M1→M2 判定: `audit/m1_accepted_parent.json`
- M2→M3 判定: `audit/m2_accepted_parent.json`
- 実行 notebook: `notebooks/30_m2_armillary.ipynb`
- M3 notebook: `notebooks/40_m3_gpu_triad_atrg.ipynb`
- roadmap: `validated_4d_su2_rg_full_plan_bundle/M1_M6_VALIDATED_RG_ROADMAP.md`
- certification rules: `validated_4d_su2_rg_full_plan_bundle/MATHEMATICAL_CERTIFICATION_SPEC.md`


In [ ]:
from pathlib import Path
import hashlib
import json

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

audit_path = PROJECT_ROOT / 'audit/m2_accepted_parent.json'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
paths = {
    'M2 report': Path(audit['m2_report_path']),
    'M2 acceptance': Path(audit['m2_acceptance_path']),
    'M2 manifest': Path(audit['manifest_path']),
    'M2 checkpoint hashes': Path(audit['checkpoint_path']) / 'hashes.json',
}
expected = {
    'M2 report': audit['m2_report_sha256'],
    'M2 acceptance': audit['m2_acceptance_sha256'],
    'M2 manifest': audit['manifest_sha256'],
    'M2 checkpoint hashes': audit['checkpoint_hash_manifest_sha256'],
}
checks = {}
for label, path in paths.items():
    digest = hashlib.sha256(path.read_bytes()).hexdigest() if path.is_file() else None
    checks[label] = {
        'path': str(path), 'sha256': digest,
        'expected': expected[label], 'PASS': digest == expected[label],
    }
if not all(item['PASS'] for item in checks.values()):
    raise RuntimeError('M2参照artifactが欠落または変更されています。M3を開始できません。')
print(json.dumps({
    '判定': audit['human_readable_decision_ja'],
    'artifact_checks': checks,
    'review_checks': audit['review_checks'],
    'certification_status': audit['certification_status'],
}, ensure_ascii=False, indent=2))


## 6時間停止後の再開

M3を人間が再開する場合:

```bash
export VALIDATED_RG_PERSIST_ROOT=/storage/validated_4d_su2_rg
export VALIDATED_RG_PERSIST_ACK=I_CONFIRM_THIS_PATH_IS_PERSISTENT
export VALIDATED_RG_M2_RUN_ID=M2-20260720T005145Z-dd3e385d0a61
export VALIDATED_RG_M3_RUN_ID=M3-20260720T013551Z-ae995e91e861
```

fresh kernel で `notebooks/40_m3_gpu_triad_atrg.ipynb` を上から実行します。15分周期、phase境界、基底変更前後にatomic checkpointが保存されます。5時間以降は長いtaskを開始せず、5時間20分までにGPU stateをCPUへ戻して最終保存を開始し、5時間30分までに終了します。

GitHubへの安全保存はテスト済み区切りごとに行います。

```bash
git status
git push origin main
```
